In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

matrix_1_path = 'JHPS2023data_ver2.0_FTA-Energy_deleted.csv'
matrix_1 = pd.read_csv(matrix_1_path, header=None)

matrix_2_path = 'JHPS2023newcohort-B_data_ver2.0_FTA-Energy_deleted.csv'
matrix_2 = pd.read_csv(matrix_2_path, header=None)

matrix_detail_path = 'JHPS2023codebook_ver2.1.xlsx'
matrix_detail = pd.read_excel(matrix_detail_path)

matrix = pd.concat([matrix_1, matrix_2], ignore_index=True)

#matrixから, 使用する列のみを選択する
matrix = matrix.loc[:, [3, 4, 5, 8, 9, 11, 13, 15, 16, 17, 19, 20, 556, 557, 564, 587, 610, 611, 612, 614, 615, 618, 627, 634, 637, 643, 651, 652, 675, 677, 690, 692, 707, 769, 782, 808, 811, 853, 872, 907, 908, 1249, 1589, 1590]]

column_list = []
for index in range(len(matrix_detail['項目'])):
    if ( pd.isna( matrix_detail['ｓｕｂ項目'].iloc[index] ) ) or (matrix_detail['ｓｕｂ項目'].iloc[index] == '　'):
        column_list.append( str( matrix_detail['項目'].iloc[index] ) )
    else:
        column_list.append( str( matrix_detail['項目'].iloc[index] )+ '（' + str( matrix_detail['ｓｕｂ項目'].iloc[index] ) + '）' )

column_list = [column_list[i] for i in [3, 4, 5, 8, 9, 11, 13, 15, 16, 17, 19, 20, 556, 557, 564, 587, 610, 611, 612, 614, 615, 618, 627, 634, 637, 643, 651, 652, 675, 677, 690, 692, 707, 769, 782, 808, 811, 853, 872, 907, 908, 1249, 1589, 1590]]

#カラム名の重複を回避するために, column_listを改良
column_list[6] = '世帯員が単身赴任から戻る'
column_list[7] = '世帯員が単身赴任する'
column_list[34] = '本人の幸福感（最近１年間）'
column_list[41] = '配偶者の幸福感（最近１年間）'

matrix.columns = column_list

#いくつかの属性について、とりうる値の数を少なくすることでデータサイズを小さくする

#対象者生年月日（生年）
filtered = matrix['対象者生年月日（生年）'].copy()

binned_data = pd.qcut(filtered, q=10, duplicates='drop')
transformed_data = binned_data.apply(lambda x: x.left)
result = matrix['対象者生年月日（生年）'].copy()
result.loc[filtered.index] = transformed_data.astype(float).round(0).astype(int)

matrix['対象者生年月日（生年）'] = result

#両親への経済援助
exclude_values = [99999, 88888]
filtered = matrix['両親への経済援助'][~matrix['両親への経済援助'].isin(exclude_values)]

binned_data = pd.qcut(filtered, q=10, duplicates='drop')
transformed_data = binned_data.apply(lambda x: x.left)
result = matrix['両親への経済援助'].copy()
result.loc[filtered.index] = transformed_data.astype(float).round(0).astype(int)

matrix['両親への経済援助'] = result

#両親からの経済援助
exclude_values = [99999, 88888]
filtered = matrix['両親からの経済援助'][~matrix['両親からの経済援助'].isin(exclude_values)]

binned_data = pd.qcut(filtered, q=10, duplicates='drop')
transformed_data = binned_data.apply(lambda x: x.left)
result = matrix['両親からの経済援助'].copy()
result.loc[filtered.index] = transformed_data.astype(float).round(0).astype(int)

matrix['両親からの経済援助'] = result

#仕事からの収入（昨年）
exclude_values = [99999, 88888]
filtered = matrix['仕事からの収入（昨年）'][~matrix['仕事からの収入（昨年）'].isin(exclude_values)]

binned_data = pd.qcut(filtered, q=10, duplicates='drop')
transformed_data = binned_data.apply(lambda x: x.left)
result = matrix['仕事からの収入（昨年）'].copy()
result.loc[filtered.index] = transformed_data.astype(float).round(0).astype(int)

matrix['仕事からの収入（昨年）'] = result

#週平均残業時間
exclude_values = [999, 888]
filtered = matrix['週平均残業時間'][~matrix['週平均残業時間'].isin(exclude_values)]

binned_data = pd.qcut(filtered, q=10, duplicates='drop')
transformed_data = binned_data.apply(lambda x: x.left)
result = matrix['週平均残業時間'].copy()
result.loc[filtered.index] = transformed_data.astype(float).round(0).astype(int)

matrix['週平均残業時間'] = result

#平日睡眠時間（平均時間）
exclude_values = [999.9]
filtered = matrix['平日睡眠時間（平均時間）'][~matrix['平日睡眠時間（平均時間）'].isin(exclude_values)]

binned_data = pd.qcut(filtered, q=10, duplicates='drop')
transformed_data = binned_data.apply(lambda x: x.left)
result = matrix['平日睡眠時間（平均時間）'].copy()
result.loc[filtered.index] = transformed_data.astype(float).round(0).astype(int)

matrix['平日睡眠時間（平均時間）'] = result

#休日睡眠時間（平均時間）
exclude_values = [999.9]
filtered = matrix['休日睡眠時間（平均時間）'][~matrix['休日睡眠時間（平均時間）'].isin(exclude_values)]

binned_data = pd.qcut(filtered, q=10, duplicates='drop')
transformed_data = binned_data.apply(lambda x: x.left)
result = matrix['休日睡眠時間（平均時間）'].copy()
result.loc[filtered.index] = transformed_data.astype(float).round(0).astype(int)

matrix['休日睡眠時間（平均時間）'] = result

#配偶者の幸福感（最近１年間）
exclude_values = [99, 88]
filtered = matrix['配偶者の幸福感（最近１年間）'][~matrix['配偶者の幸福感（最近１年間）'].isin(exclude_values)]

binned_data = pd.qcut(filtered, q=5, duplicates='drop')
transformed_data = binned_data.apply(lambda x: x.left)
result = matrix['配偶者の幸福感（最近１年間）'].copy()
result.loc[filtered.index] = transformed_data.astype(float).round(0).astype(int)

matrix['配偶者の幸福感（最近１年間）'] = result

#欠損値リスト・非該当値リストを作成する（dictionaryやmatrix_fpgrowthの作成で使用する）
missing_list = []
missing_list = matrix_detail['無回答'].iloc[[3, 4, 5, 8, 9, 11, 13, 15, 16, 17, 19, 20, 556, 557, 564, 587, 610, 611, 612, 614, 615, 618, 627, 634, 637, 643, 651, 652, 675, 677, 690, 692, 707, 769, 782, 808, 811, 853, 872, 907, 908, 1249, 1589, 1590]].to_list()
unapplicable_list = []
unapplicable_list = matrix_detail['非該当'].iloc[[3, 4, 5, 8, 9, 11, 13, 15, 16, 17, 19, 20, 556, 557, 564, 587, 610, 611, 612, 614, 615, 618, 627, 634, 637, 643, 651, 652, 675, 677, 690, 692, 707, 769, 782, 808, 811, 853, 872, 907, 908, 1249, 1589, 1590]].to_list()

#辞書の作成
dictionary = {}
for column_name in matrix.columns:
  unique_values_array = matrix[column_name].unique()
  unique_values_list = unique_values_array.tolist()
  dictionary[column_name] = unique_values_list

#辞書の各キーの値から欠損値を削除
for i in range(len(missing_list)):
  if pd.isna(missing_list[i]):
    continue
  else:
    for item_to_remove in list(dictionary.values())[i]:
      if item_to_remove == missing_list[i]:
        dictionary[list(dictionary.keys())[i]].remove(item_to_remove)

#辞書の各キーの値から非該当値を削除
for i in range(len(unapplicable_list)):
  if pd.isna(unapplicable_list[i]):
    continue
  else:
    for item_to_remove in list(dictionary.values())[i]:
      if item_to_remove == unapplicable_list[i]:
        dictionary[list(dictionary.keys())[i]].remove(item_to_remove)

matrix_train, matrix_test = train_test_split(matrix, test_size=0.1, random_state=42)

#fpgrowth関数に入力するために、matrix_trainをダミー変数化
matrix_fpgrowth = pd.get_dummies(matrix_train, columns=list(dictionary.keys()))

#matrix_aprioriから欠損値を示す列を削除
drop_columns = []
for i in range(len(missing_list)):
    if pd.isna(missing_list[i]):
        continue
    else:
        drop_column = str( list(dictionary.keys())[i] ) + '_' + str( int(missing_list[i]) )
        if drop_column in matrix_fpgrowth.columns:
            matrix_fpgrowth = matrix_fpgrowth.drop(drop_column, axis=1)

#matrix_fpgrowthから非該当値を示す列を削除
drop_columns_2 = []
for i in range(len(unapplicable_list)):
    if pd.isna(unapplicable_list[i]):
        continue
    else:
        drop_column_2 = str( list(dictionary.keys())[i] ) + '_' + str( int(unapplicable_list[i]) )
        if drop_column_2 in matrix_fpgrowth.columns:
            matrix_fpgrowth = matrix_fpgrowth.drop(drop_column_2, axis=1)

matrix_fpgrowth.head()

,配偶者の有無_1,配偶者の有無_2,対象者性別_1,対象者性別_2,対象者生年月日（生年）_1929,対象者生年月日（生年）_1944,対象者生年月日（生年）_1950,対象者生年月日（生年）_1956,対象者生年月日（生年）_1963,対象者生年月日（生年）_1969,...,地域ブロック_2,地域ブロック_3,地域ブロック_4,地域ブロック_5,地域ブロック_6,地域ブロック_7,地域ブロック_8,市郡規模_1,市郡規模_2,市郡規模_3
2175,False,True,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,True,False,True,False
817,True,False,False,True,False,True,False,False,False,False,...,False,False,False,False,True,False,False,False,True,False
296,True,False,False,True,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,True,False
1360,False,True,True,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,True,False
410,True,False,True,False,True,False,False,False,False,False,...,False,False,False,False,False,False,True,False,True,False


In [2]:
# データから相関ルールを学習する
min_support=0.05 #今回の設定①
frequent_itemsets = fpgrowth(matrix_fpgrowth, min_support=min_support, use_colnames=True, max_len=4)
rules = association_rules(frequent_itemsets, min_threshold=0.1)

In [3]:
rules.shape

(633988, 14)

In [4]:
#rulesの内, 結論部が「最近1年間の幸福度」のものを抽出
target_features = [
    frozenset({'本人の幸福感（最近１年間）_0'}),
    frozenset({'本人の幸福感（最近１年間）_1'}),
    frozenset({'本人の幸福感（最近１年間）_2'}),
    frozenset({'本人の幸福感（最近１年間）_3'}),
    frozenset({'本人の幸福感（最近１年間）_4'}),
    frozenset({'本人の幸福感（最近１年間）_5'}),
    frozenset({'本人の幸福感（最近１年間）_6'}),
    frozenset({'本人の幸福感（最近１年間）_7'}),
    frozenset({'本人の幸福感（最近１年間）_8'}),
    frozenset({'本人の幸福感（最近１年間）_9'}),
    frozenset({'本人の幸福感（最近１年間）_10'})
]

rules = rules[rules['consequents'].isin(target_features)]

In [5]:
rules.shape

(1599, 14)

In [6]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
293070,(休日睡眠時間（平均時間）_2.0),(本人の幸福感（最近１年間）_5),0.209379,0.213643,0.054950,0.262443,1.228422,1.0,0.010218,1.066165,0.235191,0.149292,0.062059,0.259825
297377,"(一年前の居住_1, 休日睡眠時間（平均時間）_2.0)",(本人の幸福感（最近１年間）_5),0.200853,0.213643,0.054003,0.268868,1.258493,1.0,0.011092,1.075534,0.257022,0.149803,0.070229,0.260820
362998,(一年前の居住_1),(本人の幸福感（最近１年間）_5),0.946945,0.213643,0.207958,0.219610,1.027930,1.0,0.005650,1.007646,0.512122,0.218299,0.007588,0.596501
363001,(介護を必要とする家族_4),(本人の幸福感（最近１年間）_5),0.816201,0.213643,0.176694,0.216483,1.013293,1.0,0.002318,1.003625,0.071377,0.207107,0.003612,0.521767
363002,(市郡規模_2),(本人の幸福感（最近１年間）_5),0.620559,0.213643,0.143060,0.230534,1.079064,1.0,0.010482,1.021952,0.193103,0.206991,0.021481,0.450079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540623,"(飲酒習慣_1, 一年前の居住_1)",(本人の幸福感（最近１年間）_6),0.469919,0.133112,0.062056,0.132056,0.992068,1.0,-0.000496,0.998784,-0.014859,0.114711,-0.001218,0.299124
540626,"(技術・技能の習得_3, 飲酒習慣_1)",(本人の幸福感（最近１年間）_6),0.421601,0.133112,0.057793,0.137079,1.029797,1.0,0.001672,1.004596,0.050026,0.116301,0.004575,0.285621
540633,"(飲酒習慣_1, 介護を必要とする家族_4)",(本人の幸福感（最近１年間）_6),0.392705,0.133112,0.053529,0.136309,1.024014,1.0,0.001255,1.003701,0.038615,0.113340,0.003687,0.269222
540638,"(飲酒習慣_1, タバコの喫煙_4)",(本人の幸福感（最近１年間）_6),0.342965,0.133112,0.052582,0.153315,1.151771,1.0,0.006929,1.023861,0.200556,0.124161,0.023305,0.274166


相関ルールマイニングを踏まえて推薦を行う

In [7]:
#準備① : rulesをもとに, 各ルールの前提部・結論部(幸福度)・確信度から成るpd.Dataframe「rules_df」を作成

#まずはDataFrameではなくリストを作成
rules_list = []
for index, row in rules[['antecedents', 'consequents', 'confidence']].iterrows():
  antecedent = list(row['antecedents'])
  consequent = list(row['consequents'])
  confidence = row['confidence']
  rules_list.append([antecedent, consequent, confidence]) #3つの要素をリストにして最終的なリストに追加

print(rules_list)

[[['休日睡眠時間（平均時間）_2.0'], ['本人の幸福感（最近１年間）_5'], 0.26244343891402716], [['一年前の居住_1', '休日睡眠時間（平均時間）_2.0'], ['本人の幸福感（最近１年間）_5'], 0.2688679245283019], [['一年前の居住_1'], ['本人の幸福感（最近１年間）_5'], 0.21960980490245124], [['介護を必要とする家族_4'], ['本人の幸福感（最近１年間）_5'], 0.21648287869994196], [['市郡規模_2'], ['本人の幸福感（最近１年間）_5'], 0.23053435114503815], [['タバコの喫煙_4'], ['本人の幸福感（最近１年間）_5'], 0.19544740973312405], [['両親への経済援助_0'], ['本人の幸福感（最近１年間）_5'], 0.20179007323026849], [['対象者性別_1'], ['本人の幸福感（最近１年間）_5'], 0.2304725168756027], [['飲酒習慣_1'], ['本人の幸福感（最近１年間）_5'], 0.21944177093359002], [['両親の生死_1'], ['本人の幸福感（最近１年間）_5'], 0.18677494199535963], [['地域ブロック_3'], ['本人の幸福感（最近１年間）_5'], 0.20704225352112676], [['配偶者の有無_2'], ['本人の幸福感（最近１年間）_5'], 0.20359281437125748], [['技術・技能の習得_3'], ['本人の幸福感（最近１年間）_5'], 0.22761849034523113], [['通勤通学以外で運動する日数_8'], ['本人の幸福感（最近１年間）_5'], 0.21755438859714926], [['対象者性別_2'], ['本人の幸福感（最近１年間）_5'], 0.19739292364990688], [['両親の生死_4'], ['本人の幸福感（最近１年間）_5'], 0.23355704697986576], [['１年前の就業_8'], ['本人の幸福感（最近１年間）_5'], 0.

In [8]:
flattened_rules_list = []
for rule in rules_list:
  flattened_rules_list_ = []
  for rule_ in rule:
    if isinstance(rule_, list):
      flattened_rules_list_.extend(rule_)
    else:
      flattened_rules_list_.append(rule_)
  flattened_rules_list.append(flattened_rules_list_)

rules_list = flattened_rules_list

print(rules_list)

[['休日睡眠時間（平均時間）_2.0', '本人の幸福感（最近１年間）_5', 0.26244343891402716], ['一年前の居住_1', '休日睡眠時間（平均時間）_2.0', '本人の幸福感（最近１年間）_5', 0.2688679245283019], ['一年前の居住_1', '本人の幸福感（最近１年間）_5', 0.21960980490245124], ['介護を必要とする家族_4', '本人の幸福感（最近１年間）_5', 0.21648287869994196], ['市郡規模_2', '本人の幸福感（最近１年間）_5', 0.23053435114503815], ['タバコの喫煙_4', '本人の幸福感（最近１年間）_5', 0.19544740973312405], ['両親への経済援助_0', '本人の幸福感（最近１年間）_5', 0.20179007323026849], ['対象者性別_1', '本人の幸福感（最近１年間）_5', 0.2304725168756027], ['飲酒習慣_1', '本人の幸福感（最近１年間）_5', 0.21944177093359002], ['両親の生死_1', '本人の幸福感（最近１年間）_5', 0.18677494199535963], ['地域ブロック_3', '本人の幸福感（最近１年間）_5', 0.20704225352112676], ['配偶者の有無_2', '本人の幸福感（最近１年間）_5', 0.20359281437125748], ['技術・技能の習得_3', '本人の幸福感（最近１年間）_5', 0.22761849034523113], ['通勤通学以外で運動する日数_8', '本人の幸福感（最近１年間）_5', 0.21755438859714926], ['対象者性別_2', '本人の幸福感（最近１年間）_5', 0.19739292364990688], ['両親の生死_4', '本人の幸福感（最近１年間）_5', 0.23355704697986576], ['１年前の就業_8', '本人の幸福感（最近１年間）_5', 0.23423423423423426], ['休日睡眠時間（平均時間）_7.0', '本人の幸福感（最近１年間）_5', 0.188940

In [9]:
#各ルールの前提部, 結論部(幸福度), 確信度を示すリストを, DataFrame化
rules_df = pd.Series(index=list(dictionary.keys()) + ['確信度'], dtype='object') #空のSeriesを作成
rules_df = rules_df.to_frame().T

for i in range(len(rules_list)):
  rules_df_concat = pd.Series(index=list(dictionary.keys()) + ['確信度'], dtype='object')
  rules_df_concat = rules_df_concat.to_frame().T

  for rule_ in rules_list[i]:
    if isinstance(rule_, str) and '_' in rule_:
      rule_ = rule_.split('_')
      rule__column = rule_[0]
      rule__value = int( float( rule_[1] ) )
      rules_df_concat[rule__column] = rule__value
    else:
      rules_df_concat['確信度'] = rule_

  rules_df = pd.concat([rules_df, rules_df_concat])

rules_df = rules_df.iloc[1:]

rules_df.head()

,配偶者の有無,対象者性別,対象者生年月日（生年）,同居人数,一年前の居住,世帯変動・子ども,世帯員が単身赴任から戻る,世帯員が単身赴任する,世帯変動・転出,世帯変動・死亡,...,飲酒習慣,タバコの喫煙,通勤通学以外で運動する日数,介護を必要とする家族,平日睡眠時間（平均時間）,休日睡眠時間（平均時間）,配偶者の幸福感（最近１年間）,地域ブロック,市郡規模,確信度
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,0.262443
0,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,0.268868
0,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.219610
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,0.216483
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.230534


In [153]:
#準備② : 「profile_df」と, それに対応するworried_columnを作成

profile_set = matrix_test.copy()

profile_df = pd.Series(index=list(dictionary.keys()), dtype='object') #空のSeriesを作成
profile_df = profile_df.to_frame().T

profile_df_append = profile_set.iloc[0, :] #今回の設定②
#profileに欠損値や非該当値が含まれている場合, Noneに置換する
for i in range(len(profile_df_append)):
    if profile_df_append.iloc[i] in dictionary[ list(dictionary.keys())[i] ]:
        continue
    else:
        profile_df_append.iloc[i] = None
profile_df_append = profile_df_append.to_frame().T

profile_df = pd.concat([profile_df, profile_df_append])
profile_df = profile_df.iloc[1:]

worried_column = '配偶者の有無' #今回の設定③

/tmp/ipykernel_17695/4268669440.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  profile_df_append.iloc[i] = None


In [154]:
profile_df.head()

,配偶者の有無,対象者性別,対象者生年月日（生年）,同居人数,一年前の居住,世帯変動・子ども,世帯員が単身赴任から戻る,世帯員が単身赴任する,世帯変動・転出,世帯変動・死亡,...,本人の幸福感（最近１年間）,飲酒習慣,タバコの喫煙,通勤通学以外で運動する日数,介護を必要とする家族,平日睡眠時間（平均時間）,休日睡眠時間（平均時間）,配偶者の幸福感（最近１年間）,地域ブロック,市郡規模
1404,2.0,2.0,1989.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,3.0,5.0,1.0,8.0,4.0,7.0,9.0,NaN,3.0,1.0


In [ ]:
#推定幸福度の準備 : happiness_estimationを実行するための決まった処理

#rules_dfの, 幸福度と確信度以外の列から成る「rules_df_rules」を作成する
rules_df_rules = rules_df.drop(columns=['本人の幸福感（最近１年間）', '確信度'])

#各ルールの確信度が格納されたリスト「confidence_list」を作成する
rules_df_confidence = rules_df['確信度']
confidence_list = rules_df_confidence.tolist()
confidence_list = [min(0.7, conf) for conf in confidence_list]

In [156]:
def happiness_estimation(profile_df):
  #各ルールのプロフィールとのマッチ度が格納されたリスト「matching_list」を作成する
  matching_list = []
  profile_df_main = profile_df.drop(columns=['本人の幸福感（最近１年間）'])
  
  for index, rules_df_rules_ in rules_df_rules.iterrows():
    a = []
    #a*wの計算を, aとwのcountが少ない方の数だけ行う
    a_and_w_count = min( int(rules_df_rules_.count()), int(profile_df_main.count(axis=1)) )
    w = [1 / a_and_w_count] * a_and_w_count
    for i in range(profile_df_main.shape[1]): #43
      if pd.notna( rules_df_rules_.iloc[i] ) and pd.notna( profile_df_main.iloc[0, i] ):
        
        a_under = max( list(dictionary.values())[i] ) - min( list(dictionary.values())[i] )
        a_upper_1 = rules_df_rules_.iloc[i]
        a_upper_2 = profile_df_main.iloc[0, i]
        #ただし, 属性が「両親の生死」または「1年前の仕事との違い」の場合、profileの当該属性値を0, それ以外の属性値を1とする
        if list(dictionary.keys())[i] in ('両親の生死', '技術・技能の習得', '副業の有無', '仕事の内容', '経営組織', '職位', '働き方', '現在の仕事の継続', '仕事を変えたい理由', '１年前の就業', '通勤通学以外で運動する日数', '介護を必要とする家族', '地域ブロック', '市群規模'):
          if rules_df_rules_.iloc[i] == profile_df_main.iloc[0, i]:
            a_upper_1 = 0
            a_upper_2 = 0
          else:
            a_upper_1 = 0
            a_upper_2 = 1
        a.append( 1 - abs( (a_upper_1 / a_under) - (a_upper_2 / a_under) ))

    matching_pre = sum( [val_a * val_w for val_a, val_w in zip(a, w)] )
    matching = ( a_and_w_count / (profile_df_main.shape[1]) ) * matching_pre
    matching_list.append(matching)

  #各ルールの重要度が格納されたリスト
  importance = [confidence * matching for confidence, matching in zip(confidence_list, matching_list)]

  #推定幸福度を計算
  estimated_happiness_apper = 0 #推定幸福度を計算する加重平均の式における, 分子
  estimated_happiness_under = 0 #推定幸福度を計算する加重平均の式における, 分母
  for i in range(len(importance)):
    estimated_happiness_apper += importance[i] * rules_df.iloc[i]['本人の幸福感（最近１年間）']
    estimated_happiness_under += importance[i]

  estimated_happiness = estimated_happiness_apper / estimated_happiness_under

  return estimated_happiness

In [ ]:
#様々なprofile_dfを作成して, その度にhappiness_estimation関数を実行する
estimated_happiness = happiness_estimation(profile_df)
print(estimated_happiness)

recommendations = []

for worried_column_value in dictionary[worried_column]:
  #まずは, 悩んでいる属性の属性値を変えたときのprofileをDataFrame化する
  new_profile_df = profile_df.copy()
  new_profile_df[worried_column] = worried_column_value
  
  if new_profile_df[worried_column].iloc[0] != profile_df[worried_column].iloc[0]:
    new_estimated_happiness = happiness_estimation(new_profile_df)
    print(new_profile_df[worried_column].iloc[0])
    print(new_estimated_happiness)

    if new_estimated_happiness > estimated_happiness:
      recommendations.append(worried_column_value) #あとで, 推定幸福度が大きいものから順にrecommendationsに格納するようにする

if recommendations == []:
  print('推薦内容はありません')
else:
  print(recommendations)

6.130324254757696
1
6.136571931690092
[1]
